# 실습 14: 잣대가 둘이면 겹치는 데가 보인다
- 상황: 한 방법으로만 짚은 결과를 그대로 믿기는 어렵다
- 목표: 다르게 생긴 방법으로 한 번 더 짚고, 둘이 같이 가리킨 줄을 찾는다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 센서 열만 골라 빈칸을 그 열의 중앙값으로 채운다
센서열 = [c for c in df.columns if c.startswith("sensor_")]
df[센서열] = df[센서열].fillna(df[센서열].median())

X = df[센서열]
정답 = (df["result"] == "불량").astype(int)

# 정답은 넣지 않는다 - 센서 값만 담긴 X로 지목한다
탐지기 = IsolationForest(contamination=0.05, random_state=42)
고립지목 = 탐지기.fit_predict(X)

이상건수 = int((고립지목 == -1).sum())
이상중불량 = int(((고립지목 == -1) & (정답 == 1)).sum())

print("이상(-1)으로 지목된 건수:", 이상건수)
print("그중 불량 건수:", 이상중불량)

이상(-1)으로 지목된 건수: 79
그중 불량 건수: 12


---
## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 두 번째 잣대

| 수업에서 쓰는 말 | 정식 이름 | 뜻 |
|---|---|---|
| 이웃과 비교하기 | 국소 이상치 인자 (Local Outlier Factor, LOF) | 내 주변이 얼마나 붐비는지를 보는 방법. 혼자 떨어져 있으면 이상 |
| 이웃 수 | n_neighbors | 몇 명을 이웃으로 볼지. 이것도 사람이 정하는 값 |
| 끝값 | 이상치 (outlier) | 다른 값들보다 유난히 크거나 작은 값. 고립시키기가 잘 잡는 쪽 |
| 무리에서 떨어진 것 | 국소 이상치 (local outlier) | 값 자체는 평범한데 어느 무리에도 안 끼는 줄. 이웃 비교가 잘 잡는 쪽 |
| 겹침 | 교집합 (intersection) | 두 방법이 같이 지목한 줄 |
| 여러 방법을 합쳐 쓰기 | 앙상블 (ensemble) | 한 방법만 믿지 않고 여러 결과를 모아 판단하는 것 |

가운데 칸이 진짜 이름이다. 지난주에 배운 정밀도·재현율처럼 이쪽으로 말해야 통한다.

---
## Step 2. 이웃과 비교해서 지목하기

In [2]:
# 주변이 얼마나 붐비는지로 이상을 찾는 도구를 불러온다
from sklearn.neighbors import LocalOutlierFactor

# ① 바로 위에서 불러온 도구 이름을 그대로 쓴다
# ② 이웃을 몇 명까지 볼지 정하는 자리
# ③ 몇 %를 이상으로 볼지 정하는 자리 - 앞 실습과 같게 둬야 견줄 수 있다
이웃탐지기 = LocalOutlierFactor(n_neighbors=20, contamination=0.05)

# ④ 여기도 정답은 넣지 않는다
이웃지목 = 이웃탐지기.fit_predict(X)

# ⑤ 이 도구도 앞 실습과 같은 약속을 쓴다
이웃이상 = (이웃지목 == -1)

# ⑥ 두 표시가 둘 다 참인 자리만 남기는 기호
# ⑦ 불량을 무엇으로 적어뒀는지
print("이웃 비교 지목:", 이웃이상.sum(), "건")
print("그중 불량:", (이웃이상 & (정답 == 1)).sum(), "건")

이웃 비교 지목: 79 건
그중 불량: 7 건


### 문법 노트 - 두 번째 방법

| 밑줄 | 넣은 것 | 하는 일 | 새로 배운 것인가 |
|---|---|---|---|
| ① | `LocalOutlierFactor` | 주변이 얼마나 붐비는지로 이상을 찾는 도구 | 새로 배움 |
| ② | `n_neighbors` | 이웃을 몇 명까지 볼지 정하는 자리 | 새로 배움 |
| ③ | `contamination` | 전체의 몇 %를 이상으로 볼지 | 앞 실습에서 배웠다 |
| ④ | `X` | 센서 값만 담긴 것 | 계속 쓴 이름 |
| ⑤ | `-1` | 이상이라는 표시 (정상은 `1`) | 앞 실습에서 배웠다 |
| ⑥ | `&` | 양쪽이 다 참인 자리만 참 | 앞 실습에서 배웠다 |
| ⑦ | `1` | 불량을 1로 적어둔 그 값 | 계속 쓰는 약속 |

---
## Step 4. 두 방법을 나란히

In [3]:
# 앞에서 만든 고립지목, 이웃지목을 그대로 채점한다 - 둘 다 contamination=0.05로 같다
이웃이상 = (이웃지목 == -1)

전체불량건수 = int((정답 == 1).sum())
전체불량률 = round(정답.mean() * 100, 1)

rows = [
    {
        "방법": "고립시키기 (IsolationForest)",
        "지목 건수": 이상건수,
        "그중 불량": 이상중불량,
        "지목 중 진짜(%)": round(이상중불량 / 이상건수 * 100, 1),
        "전체 불량 중 잡은 것(%)": round(이상중불량 / 전체불량건수 * 100, 1),
    },
    {
        "방법": "이웃 비교 (LocalOutlierFactor)",
        "지목 건수": int(이웃이상.sum()),
        "그중 불량": int((이웃이상 & (정답 == 1)).sum()),
        "지목 중 진짜(%)": round(int((이웃이상 & (정답 == 1)).sum()) / int(이웃이상.sum()) * 100, 1),
        "전체 불량 중 잡은 것(%)": round(int((이웃이상 & (정답 == 1)).sum()) / 전체불량건수 * 100, 1),
    },
    {
        "방법": "전체 불량률(참고)",
        "지목 건수": "-",
        "그중 불량": 전체불량건수,
        "지목 중 진짜(%)": "-",
        "전체 불량 중 잡은 것(%)": 전체불량률,
    },
]

성적표 = pd.DataFrame(rows).set_index("방법")
성적표

,지목 건수,그중 불량,지목 중 진짜(%),전체 불량 중 잡은 것(%)
방법,,,,
고립시키기 (IsolationForest),79,12,15.2,11.5
이웃 비교 (LocalOutlierFactor),79,7,8.9,6.7
전체 불량률(참고),-,104,-,6.6


---
## Step 5. 둘이 같이 짚은 줄

In [4]:
# ⑧ 앞 실습 방식대로 고립시키기 쪽 표시도 만들어 둔다
고립이상 = (고립지목 == -1)

# ⑨ 두 방법이 같이 지목한 줄만 남기는 기호 = 교집합
둘다 = 고립이상 & 이웃이상

# ⑩ 둘 중 하나라도 지목한 줄을 남기는 기호 = 합집합
둘중하나 = 고립이상 | 이웃이상

print("둘 다 지목:", 둘다.sum(), "건 / 그중 불량", (둘다 & (정답 == 1)).sum(), "건")
print("둘 중 하나라도:", 둘중하나.sum(), "건 / 그중 불량", (둘중하나 & (정답 == 1)).sum(), "건")

# ⑪ 참인 자리가 몇 개인지 세는 것
# ⑫ 소수점 몇 자리에서 끊을지
# 교집합 중에 정답이 참인 것의 비율을 구한다
print("둘 다 지목한 것의 적중률:",
      round((둘다 & (정답 == 1)).sum() / 둘다.sum() * 100, 1), "%")

둘 다 지목: 10 건 / 그중 불량 3 건
둘 중 하나라도: 148 건 / 그중 불량 16 건
둘 다 지목한 것의 적중률: 30.0 %


### 문법 노트 - 두 목록 겹치기

두 표시를 묶는 기호가 둘이다.

    둘 다 참인 자리만    ->  &
    한쪽만 참이어도      ->  |

| 밑줄 | 넣은 것 | 하는 일 | 새로 배운 것인가 |
|---|---|---|---|
| ⑧ | `-1` | 이 도구들이 쓰는 이상 표시 | 앞 실습에서 배웠다 |
| ⑨ | `&` | 양쪽이 다 참인 자리만 참 | 앞 실습에서 배웠다 |
| ⑩ | `\|` | 한쪽이라도 참이면 참 | 새로 배움 |
| ⑪ | `.sum()` | 참인 자리의 개수를 센다 | 계속 쓴 것 |
| ⑫ | `1` | 소수점 첫째 자리까지 남긴다 | 첫날부터 쓴 `round` |

## Step 6. 오늘 알게 된 것

| 방법 | 지목 건수 | 그중 불량 | 지목 중 진짜 |
|---|---|---|---|
| 고립시키기 | [79] | [12] | [15.2%] |
| 이웃 비교 | [79] | [7] | [8.9%] |
| 둘 다 지목 | [10] | [3] | [30.0%] |

- 겹친 목록이 더 진한가 : [30.0%로 고립시키기(15.2%)보다도 높다. 혼자서는 8.9%였던 이웃 비교가 겹치니 값을 했다]
- 겹치면 무엇을 잃나 : [열어볼 건수가 79건에서 [10]건으로 줄었다. 잡은 불량도 3건뿐이라 대부분 놓친다]
- 이 목록을 어디에 쓰나 : [먼저 열어볼 순서. 여기서도 판정이 아니라 후보다]

---
## 직접 해보기 (도전) - 지난주 모델도 같은 줄을 짚었나

- 상황: 답을 보고 배운 지난주 모델과, 답 없이 짚은 오늘 목록이 같은 줄을 가리켰을까
- 할 일: 지난주 모델의 불량 예측과 오늘 겹침 목록을 다시 겹쳐본다
- 결과물: 세 줄짜리 표 1개 + 한 줄 메모

### 1단. 지난주 모델과 겹쳐 보기

In [5]:
# 지난주(day04 lab10) 모델을 그대로 재현한다 - 표준화 + class_weight="balanced"
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

지난주모델 = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

# 오늘은 나누지 않고 전체로 학습시키고 전체를 예측한다
지난주모델.fit(X, 정답)
모델예측 = 지난주모델.predict(X)
모델불량표시 = (모델예측 == 1)

# 오늘 겹침 목록(둘 다 지목)은 Step 5의 `둘다`를 그대로 쓴다
모델과겹침 = 모델불량표시 & 둘다

rows = [
    {
        "목록": "지난주 모델이 불량이라 한 것",
        "지목 건수": int(모델불량표시.sum()),
        "그중 불량": int((모델불량표시 & (정답 == 1)).sum()),
    },
    {
        "목록": "오늘 겹침 목록 (둘 다 지목)",
        "지목 건수": int(둘다.sum()),
        "그중 불량": int((둘다 & (정답 == 1)).sum()),
    },
    {
        "목록": "지난주 모델 ∩ 오늘 겹침 목록",
        "지목 건수": int(모델과겹침.sum()),
        "그중 불량": int((모델과겹침 & (정답 == 1)).sum()),
    },
]

for r in rows:
    r["지목 중 진짜(%)"] = round(r["그중 불량"] / r["지목 건수"] * 100, 1) if r["지목 건수"] > 0 else 0.0

비교표 = pd.DataFrame(rows).set_index("목록")
비교표

,지목 건수,그중 불량,지목 중 진짜(%)
목록,,,
지난주 모델이 불량이라 한 것,427,76,17.8
오늘 겹침 목록 (둘 다 지목),10,3,30.0
지난주 모델 ∩ 오늘 겹침 목록,10,3,30.0


### 답을 본 쪽과 안 본 쪽

| 무엇 | 지목 건수 | 그중 불량 | 지목 중 진짜 |
|---|---|---|---|
| 지난주 모델 (답을 보고 배움) | [427] | [76] | [17.8%] |
| 오늘 겹침 목록 (답 안 봄) | [10] | [3] | [30.0%] |
| 셋이 다 가리킨 줄 | [10] | [3] | [30.0%] |

- 알게 된 것 :
<br>1.완전 포함 관계 — 오늘 두 이상탐지 방법(고립시키기 + 이웃 비교)이 같이 지목한 10건은 100% 지난주 로지스틱 회귀 모델도 불량으로 예측했습니다. 서로 다른 원리(답 없이 이상만 보는 방법 vs 답을 보고 배운 모델)가 같은 10건에서 만난 셈이라, 이 10건은 "여러 방법이 동시에 의심하는" 가장 신뢰도 높은 후보로 볼 수 있습니다. <br><br>2. 좁힐수록 진해진다 — 적중률(지목 중 진짜)이 단일 방법 8.9~15.2% → 두 이상탐지 겹침 30.0%로 뛰었습니다. 방법을 하나씩 더 겹칠 때마다(AND 조건) 지목 건수는 급격히 줄지만(79건→10건) 그 안의 불량 비율은 올라갑니다. 다만 대신 놓치는 불량도 많아집니다(전체 104건 중 겹침 목록은 3건만 잡음). <br><br>3. 모델과 이상탐지는 커버리지가 다르다 — 지난주 모델은 427건을 불량으로 찍어(적중률 17.8%) 훨씬 넓게 의심하는 반면, 이상탐지 겹침은 10건으로 훨씬 좁게 찍습니다. 이상탐지 겹침 10건은 모델이 넓게 찍은 427건 중에서도 "여러 근거로 동시에 의심되는" 부분집합이라는 뜻입니다.

### 2단. 이웃 수를 바꿔보기

In [6]:
# 이웃 수만 바꿔가며 이웃 비교(LOF)를 다시 돌린다 - contamination은 0.05로 고정
이웃수목록 = [5, 20, 50, 100]

결과 = []
for 이웃수 in 이웃수목록:
    탐지기_n = LocalOutlierFactor(n_neighbors=이웃수, contamination=0.05)
    지목_n = 탐지기_n.fit_predict(X)
    이상_n = (지목_n == -1)

    지목건수 = int(이상_n.sum())
    그중불량 = int((이상_n & (정답 == 1)).sum())
    겹친건수 = int((이상_n & 고립이상).sum())

    결과.append({
        "이웃 수": 이웃수,
        "지목 건수": 지목건수,
        "그중 불량": 그중불량,
        "지목 중 진짜(%)": round(그중불량 / 지목건수 * 100, 1) if 지목건수 > 0 else 0.0,
        "고립시키기와 겹친 건수": 겹친건수,
    })

이웃수표 = pd.DataFrame(결과).set_index("이웃 수")
이웃수표

,지목 건수,그중 불량,지목 중 진짜(%),고립시키기와 겹친 건수
이웃 수,,,,
5,79,4,5.1,17
20,79,7,8.9,10
50,79,12,15.2,37
100,79,11,13.9,38


### 이웃 수를 바꾸면

(강사 정제본 실측: 5는 14건, 20은 9건, 50은 42건, 100은 43건이 겹쳤다)

| 이웃 수 | 그중 불량 | 지목 중 진짜 | 고립시키기와 겹친 건수 |
|---|---|---|---|
| 5 | [4] | [5.1%] | [17] |
| 20 | [7] | [8.9]%] | [10] |
| 50 | [12] | [15.2%] | [37] |
| 100 | [11] | [13.9%] | [38] |

- 알게 된 것 : [이웃을 50명까지 보게 하니 적중이 5.1%에서 15.2%로 세 배가 됐다. 겹친 건수도 17건에서 37건으로 뛰었다]